In [72]:
import pandas as pd
import numpy as np
import random
import sklearn.manifold as manifold
import colorsys

import plotly.express as px

In [73]:
# Read distance matrix
dist_mat = pd.read_csv("../output/handshape_distance_matrix.csv")

dist_mat.head()

,Name,fbx\M20241107_6263_260710_2.fbx_segment_0,fbx\M20241107_6263_260710_2.fbx_segment_1,fbx\M20241107_6263_260710_2.fbx_segment_2,fbx\M20241107_6263_260710_2.fbx_segment_3,fbx\M20250916_4405_260709_0.fbx_segment_0,fbx\M20260119_1078_260713_1.fbx_segment_0,fbx\M20260119_1078_260713_1.fbx_segment_1,fbx\M20260119_1078_260713_1.fbx_segment_2,fbx\M20260119_1078_260713_1.fbx_segment_3,...,../HandformSession/HandformsTrimmed/Q.FBX,../HandformSession/HandformsTrimmed/R.FBX,../HandformSession/HandformsTrimmed/S.FBX,../HandformSession/HandformsTrimmed/Shh.FBX,../HandformSession/HandformsTrimmed/T.FBX,../HandformSession/HandformsTrimmed/T_Open.FBX,../HandformSession/HandformsTrimmed/V.FBX,../HandformSession/HandformsTrimmed/V_Curved.FBX,../HandformSession/HandformsTrimmed/W.FBX,../HandformSession/HandformsTrimmed/Y.FBX
0,fbx\M20241107_6263_260710_2.fbx_segment_0,0.000000,0.012738,0.035020,0.008626,0.008708,0.011029,0.030402,0.023579,0.024402,...,0.054999,0.043909,0.085492,0.025866,0.037211,0.040139,0.033062,0.054031,0.034271,0.038790
1,fbx\M20241107_6263_260710_2.fbx_segment_1,0.012738,0.000000,0.073406,0.015229,0.016588,0.015067,0.067292,0.054771,0.047117,...,0.101533,0.060356,0.140445,0.010197,0.031493,0.027318,0.049290,0.082835,0.046272,0.075073
2,fbx\M20241107_6263_260710_2.fbx_segment_2,0.035020,0.073406,0.000000,0.026328,0.024614,0.030790,0.002143,0.005242,0.013014,...,0.013708,0.035056,0.017679,0.077221,0.067934,0.083338,0.028207,0.025183,0.048380,0.018981
3,fbx\M20241107_6263_260710_2.fbx_segment_3,0.008626,0.015229,0.026328,0.000000,0.000260,0.000880,0.024674,0.018763,0.014144,...,0.049660,0.027278,0.071559,0.016052,0.019973,0.024369,0.021826,0.037611,0.033452,0.033574
4,fbx\M20250916_4405_260709_0.fbx_segment_0,0.008708,0.016588,0.024614,0.000260,0.000000,0.000639,0.023560,0.018665,0.013478,...,0.047286,0.027796,0.067524,0.017401,0.019494,0.024674,0.022410,0.036319,0.033076,0.031033


In [ ]:
# Top 10 most similar handshapes to 'fbx\M20241107_6263_260710_2.fbx_segment_2'

# Get the row for the specified handshape
# Get file name from index 1
file_name = dist_mat.index[1]
handshape_row = dist_mat.iloc[1]

# Get the indices of the 10 smallest distances
handshape_distances = handshape_row[1:].to_numpy()
similar_indices = handshape_distances.argsort()

# Get the names of the 10 most similar handshapes (excluding any names starting with 'fbx\')
similar_handshape_names = dist_mat.columns[similar_indices + 1]  # +1 to account for the first column being the name
similar_handshape_names = [name for name in similar_handshape_names if not name.startswith('fbx\\')]

print("10 most similar handshapes to:")
for handshape in similar_handshape_names:
    print(handshape)

10 most similar handshapes to:
fbx\M20241107_6263_260710_2.fbx_segment_1
fbx\M20260506_0594_260713_4.fbx_segment_1
fbx\M20260120_1291_260713_0.fbx_segment_5
fbx\M20260506_0594_260713_4.fbx_segment_10
fbx\M20260126_1564_260713_1.fbx_segment_7
fbx\M20260120_1291_260713_0.fbx_segment_1
fbx\M20260119_1078_260713_1.fbx_segment_5
fbx\M20260120_1291_260713_0.fbx_segment_7
fbx\M20260506_0594_260713_4.fbx_segment_9
fbx\M20260506_0604_260713_0.fbx_segment_0


In [75]:
# Get distance array
dist_mat = dist_mat.drop(columns=['Name'])
distance_array = dist_mat.to_numpy()

In [76]:
# # Make all distances 0 or higher
# distance_array += np.abs(np.min(distance_array))

In [77]:
# t-SNE on the distance matrix
random_num = random.randint(0, 10000)
# random_num = 8079  # This is the last one I used, so for reproducibility hardcoded
print("Using random seed:", random_num)
tsne_dist = manifold.TSNE(n_components=2, perplexity=50, learning_rate=200, init='random', metric='precomputed', method='barnes_hut', random_state=random_num)
X_tsne_dist = tsne_dist.fit_transform(distance_array)

Using random seed: 3541


In [78]:
# Dataframe for the px plot
tsne_df = pd.DataFrame({
    'tsne_1': X_tsne_dist[:, 0],
    'tsne_2': X_tsne_dist[:, 1],
    'name': dist_mat.columns,
    'is_handshape': [name.startswith("../") for name in dist_mat.columns]
})

# Plot
fig = px.scatter(
    tsne_df,
    x='tsne_1',
    y='tsne_2',
    color='is_handshape',
    hover_data=['name'],
    title="t-SNE Dimensionality Reduction of the Shape Features"
)

fig.update_layout(width=800, height=800)
fig.show()